In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tensorflow as tf

from msfm.utils import analysis

from deepsphere import healpy_layers

from deep_lss.nets.resnet import ResNetLayers
from deep_lss.models.delta_model import DeltaLossModel
from deep_lss.models.base_model import BaseModel

from msfm import fiducial_pipeline
from msfm.utils import parameters

print(tf.config.list_physical_devices("GPU"))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU')]


# constants

### strategy

In [7]:
strategy = tf.distribute.MirroredStrategy()
print(strategy.num_replicas_in_sync)

2023-03-02 02:09:39.362982: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-03-02 02:09:41.144362: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38277 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:03:00.0, compute capability: 8.0
2023-03-02 02:09:41.145727: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 38277 MB memory:  -> device: 1, name: NVIDIA A100-SXM4-40GB, pci bus id: 0000:41:00.0, compute capability: 8.0
2023-03-02 02:09:41.146906: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1532] Created device /job:localhost/replica:0/task:0/devi

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1', '/job:localhost/replica:0/task:0/device:GPU:2', '/job:localhost/replica:0/task:0/device:GPU:3')
4


### network

In [8]:
n_side = 512
n_neighbors = 8

data_vec_pix, _, _, _, _ = analysis.load_pixel_file()

23-03-02 02:09:43  analysis.py INF   Loaded the config 
23-03-02 02:09:43  analysis.py INF   Loaded the pixel file 


### dataset

In [9]:
tfr_pattern = "/pscratch/sd/a/athomsen/DESY3/v2/fiducial/DESy3_fiducial_???.tfrecord"
pert_labels = parameters.get_fiducial_perturbation_labels(["Om", "s8"])
global_batch_size = 8
n_batches = 5
input_shape = (None, len(data_vec_pix), 4)

def dataset_fn(input_context):
    dset = fiducial_pipeline.get_fiducial_dset(
        tfr_pattern=tfr_pattern,
        pert_labels=pert_labels,
        batch_size=global_batch_size,
        n_batches=n_batches,
        # relevant for performance
        n_readers=4,
        n_prefetch=1,
        file_name_shuffle_buffer=10,
        examples_shuffle_buffer=4,
        input_context=input_context,
    )

    return dset

## delta_model

### deepsphere

In [10]:
with strategy.scope():
    dist_dset = strategy.distribute_datasets_from_function(dataset_fn)

    network = ResNetLayers(output_shape=1).get_layers()

    model = DeltaLossModel(
        network=network,
        n_side=n_side,
        indices=data_vec_pix,
        n_neighbors=n_neighbors,
        input_shape=input_shape,
    )
    

# @tf.function(input_signature=[tf.TensorSpec(shape=input_shape, dtype=tf.float32)])
@tf.function
def train_step(input_batch):
    print(f"Tracing distributed train_step")
    model.distributed_train_step(
        strategy=strategy,
        input_tensor=input_batch,
        loss_function=lambda x: x,
        input_labels=None,
    )

counter = 0
for data_vectors, labels in dist_dset:
    print(f"step: {counter}")
    train_step(data_vectors)
    
    counter += 1
        
        # print(labels)

    


23-03-02 02:09:46 fiducial_pip INF   Starting to generate the fiducial training set for i_noise = 0 
23-03-02 02:09:46  analysis.py INF   Loaded the config 
23-03-02 02:09:46  analysis.py INF   Loaded the pixel file 
23-03-02 02:09:46  analysis.py INF   Loaded the config 
23-03-02 02:09:46  analysis.py INF   Loaded the pixel file 
23-03-02 02:09:46 fiducial_pip INF   Sharding the dataset over 4 replicas 
23-03-02 02:09:46 fiducial_pip INF   Using a local batch size of 4 per replica 
23-03-02 02:09:47 tfrecords.py WAR   Tracing parse_inverse_fiducial 
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: module, class, method, function, traceback, frame, or code object was expected, got cython_function_or_method
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the 

2023-03-02 02:10:12.042885: I tensorflow/stream_executor/cuda/cuda_blas.cc:1786] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2023-03-02 02:10:12.192362: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8700
2023-03-02 02:10:12.739251: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8700
2023-03-02 02:10:13.354015: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8700
2023-03-02 02:10:13.988980: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8700


step: 1
step: 2
step: 3
step: 4


### keras

In [7]:
with strategy.scope():
    dist_dset = strategy.distribute_datasets_from_function(dataset_fn)

    network = tf.keras.Sequential([
        tf.keras.layers.InputLayer((len(data_vec_pix), 4)),
        tf.keras.layers.Conv1D(7, 3),
        tf.keras.layers.Conv1D(7, 3),
        tf.keras.layers.Conv1D(7, 3),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(2),
    ])

    model = BaseModel(
        network=network,
        input_shape=input_shape,
    )
    

# @tf.function
print(input_shape)
@tf.function(input_signature=[tf.TensorSpec(shape=input_shape, dtype=tf.float32)])
def train_step(input_batch):
    print(f"Tracing distributed train_step")
    model.distributed_train_step(
        strategy=strategy,
        input_tensor=input_batch,
        loss_function=lambda x: x,
        input_labels=None,
    )

for data_vectors, labels in dist_dset:
    train_step(data_vectors)
        
    


23-03-02 01:50:34 fiducial_pip INF   Starting to generate the fiducial training set for i_noise = 0 
23-03-02 01:50:35  analysis.py INF   Loaded the config 
23-03-02 01:50:35  analysis.py INF   Loaded the pixel file 
23-03-02 01:50:35  analysis.py INF   Loaded the config 
23-03-02 01:50:35  analysis.py INF   Loaded the pixel file 
23-03-02 01:50:35 fiducial_pip INF   Sharding the dataset over 4 replicas 
23-03-02 01:50:35 fiducial_pip INF   Using a local batch size of 4 per replica 
23-03-02 01:50:35 tfrecords.py WAR   Tracing parse_inverse_fiducial 
23-03-02 01:50:35  analysis.py INF   Loaded the config 
23-03-02 01:50:35 fiducial_pip WAR   Tracing dset_add_bias 
23-03-02 01:50:35 fiducial_pip WAR   Tracing dset_add_noise 
23-03-02 01:50:35 fiducial_pip WAR   Tracing dset_concat_perts 
23-03-02 01:50:35 fiducial_pip INF   Taking 3 batches from the dataset 
23-03-02 01:50:35 fiducial_pip INF   Successfully generated the fiducial training set with element_spec (TensorSpec(shape=(20, 463

AttributeError: 'PerReplica' object has no attribute 'dtype'

## tensorflow example notebook

### keras

In [10]:
with strategy.scope():
    dist_dset = strategy.distribute_datasets_from_function(dataset_fn)

    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer((len(data_vec_pix), 4)),
        tf.keras.layers.Conv1D(7, 3),
        tf.keras.layers.Conv1D(7, 3),
        tf.keras.layers.Conv1D(7, 3),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(1),
    ])
    
    optimizer = tf.keras.optimizers.Adam()

    print(model.summary())
    
loss_object = tf.keras.losses.MeanSquaredError(reduction=tf.keras.losses.Reduction.SUM)

    
def loss_func(labels, predictions):
    per_example_loss = loss_object(labels, predictions)
    
    return per_example_loss


# loss_object = tf.keras.losses.MeanSquaredError(reduction=tf.keras.losses.Reduction.NONE)

# def loss_func(labels, predictions):
#     per_example_loss = loss_object(labels, predictions)
    
#     loss = tf.nn.compute_average_loss(per_example_loss, global_batch_size=global_batch_size)
#     return loss


23-03-02 01:52:55 fiducial_pip INF   Starting to generate the fiducial training set for i_noise = 0 
23-03-02 01:52:55  analysis.py INF   Loaded the config 
23-03-02 01:52:55  analysis.py INF   Loaded the pixel file 
23-03-02 01:52:55  analysis.py INF   Loaded the config 
23-03-02 01:52:55  analysis.py INF   Loaded the pixel file 
23-03-02 01:52:55 fiducial_pip INF   Sharding the dataset over 4 replicas 
23-03-02 01:52:55 fiducial_pip INF   Using a local batch size of 4 per replica 
23-03-02 01:52:55 tfrecords.py WAR   Tracing parse_inverse_fiducial 
23-03-02 01:52:55  analysis.py INF   Loaded the config 
23-03-02 01:52:55 fiducial_pip WAR   Tracing dset_add_bias 
23-03-02 01:52:55 fiducial_pip WAR   Tracing dset_add_noise 
23-03-02 01:52:56 fiducial_pip WAR   Tracing dset_concat_perts 
23-03-02 01:52:56 fiducial_pip INF   Taking 3 batches from the dataset 
23-03-02 01:52:56 fiducial_pip INF   Successfully generated the fiducial training set with element_spec (TensorSpec(shape=(20, 463

In [11]:
def train_step(maps, labels):
    with tf.GradientTape() as tape:
        predictions = model(maps, training=True)
        loss = loss_func(labels, predictions)

    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    return loss 

# @tf.function
@tf.function(input_signature=[
  tf.TensorSpec(shape=(None, len(data_vec_pix), 4), dtype=tf.float32),
  tf.TensorSpec(shape=(None,), dtype=tf.int32)
])
def distributed_train_step(maps, labels):
    per_replica_losses = strategy.run(train_step, args=(maps, labels))
    
    return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)


In [12]:
total_loss = 0
for maps, labels in dist_dset:
    # labels = labels[0]
    # print(labels)
    # print(model(maps))
    
    total_loss += distributed_train_step(maps, 0)
    print(total_loss)

INFO:tensorflow:batch_all_reduce: 8 all-reduces with algorithm = nccl, num_packs = 1
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


AttributeError: 'PerReplica' object has no attribute 'dtype'

### deepsphere

In [39]:
with strategy.scope():
    dist_dset = strategy.distribute_datasets_from_function(dataset_fn)

    network = ResNetLayers(output_shape=1).get_layers()

    model = DeltaLossModel(
        network=network,
        n_side=n_side,
        indices=data_vec_pix,
        n_neighbors=n_neighbors,
        input_shape=input_shape,
    )

    optimizer = tf.keras.optimizers.Adam()

    # loss_object = tf.keras.losses.MeanSquaredError(reduction=tf.keras.losses.Reduction.NONE)
    loss_object = tf.keras.losses.MeanSquaredError(reduction=tf.keras.losses.Reduction.SUM)

    def compute_loss(labels, predictions):
        per_example_loss = loss_object(labels, predictions)
        # print(per_example_loss)
        # loss = tf.nn.compute_average_loss(per_example_loss, global_batch_size=global_batch_size)
        # print(loss)
        
        # return loss
        return per_example_loss


23-03-02 01:31:00 fiducial_pip INF   Starting to generate the fiducial training set for i_noise = 0 
23-03-02 01:31:00  analysis.py INF   Loaded the config 
23-03-02 01:31:00  analysis.py INF   Loaded the pixel file 
23-03-02 01:31:00  analysis.py INF   Loaded the config 
23-03-02 01:31:00  analysis.py INF   Loaded the pixel file 
23-03-02 01:31:00 fiducial_pip INF   Sharding the dataset over 4 replicas 
23-03-02 01:31:00 fiducial_pip INF   Using a local batch size of 4 per replica 
23-03-02 01:31:00 tfrecords.py WAR   Tracing parse_inverse_fiducial 
23-03-02 01:31:00  analysis.py INF   Loaded the config 
23-03-02 01:31:00 fiducial_pip WAR   Tracing dset_add_bias 
23-03-02 01:31:00 fiducial_pip WAR   Tracing dset_add_noise 
23-03-02 01:31:00 fiducial_pip WAR   Tracing dset_concat_perts 
23-03-02 01:31:00 fiducial_pip INF   Taking 3 batches from the dataset 
23-03-02 01:31:00 fiducial_pip INF   Successfully generated the fiducial training set with element_spec (TensorSpec(shape=(20, 463

In [40]:
def train_step(maps, labels):
    with tf.GradientTape() as tape:
        predictions = model(maps, training=True)
        loss = compute_loss(labels, predictions)

    gradients = tape.gradient(loss, model.network.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.network.trainable_variables))
    
    return loss 

@tf.function
def distributed_train_step(maps, labels):
    per_replica_losses = strategy.run(train_step, args=(maps, labels))
    
    return strategy.reduce(tf.distribute.ReduceOp.SUM, per_replica_losses, axis=None)


In [41]:
total_loss = 0
for maps, labels in dist_dset:
    # labels = labels[0]
    # print(labels)
    # print(model(maps))
    
    total_loss += distributed_train_step(maps, 0)
    print(total_loss)

tf.Tensor(16.726643, shape=(), dtype=float32)
tf.Tensor(87197.17, shape=(), dtype=float32)
tf.Tensor(541942.2, shape=(), dtype=float32)
